# InsightForge AI - Fintech User Segmentation and Churn Analyzer

## Objective
In this project, I build an analytics engine that:
1. Segments fintech partner end-users by behavior (**RFM**, **lifecycle stage**, **product mix**)
2. Predicts churn probability per user and aggregates churn risk per segment
3. Surfaces **Top-3 churn drivers** as **testable hypotheses**

I structured this notebook for both model development and stakeholder communication.

## 1) Imports and Setup

I keep dependencies explicit and reproducible. Probability calibration is included to improve trust in churn probabilities (critical for segment-level decisioning).

In [22]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier


## 2) Load Fintech-Ready Dataset

The input file is generated from my event-builder pipeline and already contains:
- RFM-related features
- lifecycle-stage features
- product-mix features
- churn target window (`churned_next_30d`)

In [23]:
PROJECT_ROOT = Path(r"d:\Python\Data Science Projects\InsightForge AI")
DATA_PATH = PROJECT_ROOT / "data" / "processed_fintech_data.csv"
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)


Shape: (50000, 27)
Columns: ['user_id', 'partner_id', 'Country', 'Age', 'Gender', 'rfm_segment', 'RFM_score', 'recency_score', 'frequency_score', 'monetary_score', 'lifecycle_stage', 'tenure_days', 'days_since_last_active', 'events_last_30d', 'events_prev_30d', 'activity_trend', 'product_mix_segment', 'n_products_used', 'product_diversity', 'dominant_product', 'txn_events', 'total_amount', 'engagement_score', 'risk_score', 'support_intensity', 'segment_key', 'churned_next_30d']


,user_id,partner_id,Country,Age,Gender,rfm_segment,RFM_score,recency_score,frequency_score,monetary_score,...,n_products_used,product_diversity,dominant_product,txn_events,total_amount,engagement_score,risk_score,support_intensity,segment_key,churned_next_30d
0,76cf607b841241f6,partner_01,France,43.0,Male,At Risk,6,2,2,2,...,2,0.549815,payments,9,780.87,12.7758,3.175333,0.900000,At Risk | At Risk | Dual Product,0
1,a8505ea09744d7e2,partner_02,UK,36.0,Male,At Risk,6,1,4,1,...,3,0.779191,billpay,20,1647.33,13.6856,2.489667,0.341463,At Risk | Dormant | Multi Product,0
2,dcac8218d68b7ca7,partner_03,Canada,45.0,Female,Potential Loyalists,9,3,2,4,...,2,0.613855,investments,9,1826.93,7.2800,1.608000,0.396040,Potential Loyalists | Engaged | Dual Product,0


## 3) Data Integrity and Feature Matrix

We encode categorical variables and apply robust sanitization:
- replace inf with NaN
- median imputation for numeric columns
- quantile clipping for extreme outliers

This prevents training instability and ensures reproducible model behavior.

In [24]:
required_cols = [
    "user_id", "rfm_segment", "lifecycle_stage", "product_mix_segment",
    "segment_key", "churned_next_30d"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

feature_cols = [c for c in df.columns if c not in ["user_id", "segment_key", "churned_next_30d"]]
X_raw = df[feature_cols].copy()
y = df["churned_next_30d"].astype(int)

cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=False)

# Sanitize numeric space
X = X.replace([np.inf, -np.inf], np.nan)
for col in X.columns:
    if X[col].dtype.kind in "biufc":
        med = X[col].median()
        if pd.isna(med):
            med = 0.0
        X[col] = X[col].fillna(med)

# Clip only numeric non-boolean columns
clip_cols = [col for col in X.columns if X[col].dtype.kind in "iufc"]
if clip_cols:
    lower = X[clip_cols].quantile(0.001, numeric_only=True)
    upper = X[clip_cols].quantile(0.999, numeric_only=True)
    X.loc[:, clip_cols] = X.loc[:, clip_cols].clip(lower=lower, upper=upper, axis=1)

# Final guardrails
num_X = X.select_dtypes(include=[np.number])
if np.isinf(num_X.to_numpy(dtype=float)).any():
    raise ValueError("X still contains inf after sanitization.")
if np.isnan(num_X.to_numpy(dtype=float)).any():
    raise ValueError("X still contains NaN after sanitization.")

print("Encoded feature shape:", X.shape)
print("Class balance:")
print(y.value_counts(normalize=True).rename("ratio"))


Encoded feature shape: (50000, 58)
Class balance:
churned_next_30d
0    0.711
1    0.289
Name: ratio, dtype: float64


## 4) Split Data

I keep stratified splitting for stable churn class representation in train/test partitions.

In [25]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, X.index,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (40000, 58) Test: (10000, 58)


## 5) Baseline Model Benchmarking

I compare three practical tabular models and select the best baseline by ROC-AUC.

> Note: final deployment choice should balance both ranking power (ROC-AUC) and probability quality (Brier/LogLoss after calibration).

In [26]:
models = {
    "xgboost": XGBClassifier(
        n_estimators=450,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="logloss",
    ),
    "hist_gb": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=8,
        max_iter=400,
        random_state=42,
    ),
    "rf": RandomForestClassifier(
        n_estimators=350,
        max_depth=14,
        min_samples_split=6,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
}

results = []
trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_test)
    auc = roc_auc_score(y_test, prob)
    results.append({"model": name, "roc_auc": auc})
    trained[name] = model

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df


,model,roc_auc
0,xgboost,0.881231
1,hist_gb,0.880982
2,rf,0.865053


## 6) Probability Calibration Upgrade

### Why calibration?
Raw model scores can be over/under-confident. In churn operations, segment decisions depend on **probability quality**, not just ranking.

I calibrate the best baseline model using `CalibratedClassifierCV` and compare:
- ROC-AUC (ranking)
- Brier score (probability calibration error; lower is better)
- Log loss (probability quality; lower is better)


In [27]:
best_name = results_df.iloc[0]["model"]
best_model = trained[best_name]

# Calibrate with sigmoid for stability; isotonic can be used with very large calibration sets.
calibrated_model = CalibratedClassifierCV(best_model, method="sigmoid", cv=3)
calibrated_model.fit(X_train, y_train)

base_prob = best_model.predict_proba(X_test)[:, 1]
cal_prob = calibrated_model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame([
    {
        "model": f"{best_name}_base",
        "roc_auc": roc_auc_score(y_test, base_prob),
        "brier": brier_score_loss(y_test, base_prob),
        "log_loss": log_loss(y_test, base_prob),
    },
    {
        "model": f"{best_name}_calibrated",
        "roc_auc": roc_auc_score(y_test, cal_prob),
        "brier": brier_score_loss(y_test, cal_prob),
        "log_loss": log_loss(y_test, cal_prob),
    },
]).sort_values(["brier", "log_loss"], ascending=True)

comparison


,model,roc_auc,brier,log_loss
0,xgboost_base,0.881231,0.115001,0.371421
1,xgboost_calibrated,0.881703,0.116046,0.374916


## 7) Threshold Selection on Calibrated Probabilities

Threshold is tuned for F1 with accuracy as tie-breaker.

This gives practical classification control while preserving calibrated probabilities for segment analytics.

In [28]:
y_prob = calibrated_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.05, 0.951, 0.01)
rows = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    rows.append({
        "threshold": float(t),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
    })

thr_df = pd.DataFrame(rows)
best_thr = thr_df.sort_values(["f1", "accuracy"], ascending=False).iloc[0]
selected_threshold = float(best_thr["threshold"])

print("Selected threshold:", round(selected_threshold, 3))
best_thr


Selected threshold: 0.29


threshold    0.290000
f1           0.713782
accuracy     0.821400
precision    0.664776
recall       0.770588
Name: 24, dtype: float64

## 8) Model Evaluation Snapshot

This section reports final test metrics using calibrated probabilities + tuned threshold.

In [29]:
y_pred = (y_prob >= selected_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
metrics = {
    "roc_auc": roc_auc_score(y_test, y_prob),
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
}

pd.DataFrame([metrics])


,roc_auc,accuracy,precision,recall,f1,tn,fp,fn,tp
0,0.881703,0.8214,0.664776,0.770588,0.713782,5987,1123,663,2227


## 9) Score Full Population and Build Segment Risk Table

Now I score all users and aggregate churn risk by combined segment key:
- `rfm_segment`
- `lifecycle_stage`
- `product_mix_segment`


In [30]:
full_prob = calibrated_model.predict_proba(X)[:, 1]

scored = df.copy()
scored["churn_probability"] = full_prob
scored["churn_risk"] = np.where(scored["churn_probability"] >= selected_threshold, "High", "Low")

segment_scores = (
    scored.groupby(["rfm_segment", "lifecycle_stage", "product_mix_segment", "segment_key"], as_index=False)
    .agg(
        users=("user_id", "count"),
        avg_churn_probability=("churn_probability", "mean"),
        high_risk_users=("churn_risk", lambda x: int((x == "High").sum())),
    )
)
segment_scores["high_risk_ratio"] = segment_scores["high_risk_users"] / segment_scores["users"]
segment_scores = segment_scores.sort_values("avg_churn_probability", ascending=False)

segment_scores.head(12)


,rfm_segment,lifecycle_stage,product_mix_segment,segment_key,users,avg_churn_probability,high_risk_users,high_risk_ratio
26,Low Value,Dormant,Single Product,Low Value | Dormant | Single Product,599,0.816583,575,0.959933
9,At Risk,Dormant,Single Product,At Risk | Dormant | Single Product,546,0.784479,507,0.928571
23,Low Value,At Risk,Single Product,Low Value | At Risk | Single Product,780,0.658842,651,0.834615
5,At Risk,At Risk,Single Product,At Risk | At Risk | Single Product,1031,0.608444,798,0.774006
35,Potential Loyalists,At Risk,Single Product,Potential Loyalists | At Risk | Single Product,255,0.579645,195,0.764706
28,Low Value,Engaged,Single Product,Low Value | Engaged | Single Product,446,0.504678,295,0.661435
24,Low Value,Dormant,Dual Product,Low Value | Dormant | Dual Product,522,0.478488,313,0.599617
36,Potential Loyalists,Dormant,Dual Product,Potential Loyalists | Dormant | Dual Product,822,0.469990,482,0.586375
12,At Risk,Engaged,Single Product,At Risk | Engaged | Single Product,2626,0.425808,1444,0.549886
42,Potential Loyalists,Engaged,Single Product,Potential Loyalists | Engaged | Single Product,3453,0.393025,1737,0.503041


## 10) Top-3 Churn Drivers as Testable Hypotheses

Approach:
1. Identify segment subsets with enough samples and churn variation
2. Compare churners vs non-churners feature means within each segment
3. Extract top-3 positive driver gaps and write testable hypotheses


In [31]:
num_cols = [c for c in X.columns if X[c].dtype.kind in "biufc"]

hypothesis_rows = []
for seg, seg_df in scored.groupby("segment_key"):
    seg_idx = seg_df.index
    y_seg = y.loc[seg_idx]

    if y_seg.nunique() < 2 or len(seg_df) < 80:
        continue

    x_seg = X.loc[seg_idx, num_cols]
    churn_mean = x_seg[y_seg == 1].mean(numeric_only=True)
    stay_mean = x_seg[y_seg == 0].mean(numeric_only=True)
    gap = (churn_mean - stay_mean).sort_values(ascending=False)

    top_features = gap.head(3).index.tolist()
    for rank, feat in enumerate(top_features, start=1):
        direction = "higher" if gap[feat] > 0 else "lower"
        hypothesis_rows.append({
            "segment_key": seg,
            "rank": rank,
            "driver_feature": feat,
            "effect_gap": float(gap[feat]),
            "hypothesis": (
                f"In segment '{seg}', {direction} {feat} is associated with higher churn risk; "
                f"test a targeted intervention focused on this driver."
            ),
        })

hypotheses_df = pd.DataFrame(hypothesis_rows).sort_values(["segment_key", "rank"])
hypotheses_df.head(15)


,segment_key,rank,driver_feature,effect_gap,hypothesis
0,At Risk | Active | Multi Product,1,total_amount,189.840876,"In segment 'At Risk | Active | Multi Product',..."
1,At Risk | Active | Multi Product,2,txn_events,0.767416,"In segment 'At Risk | Active | Multi Product',..."
2,At Risk | Active | Multi Product,3,partner_id_partner_02,0.176404,"In segment 'At Risk | Active | Multi Product',..."
3,At Risk | At Risk | Dual Product,1,tenure_days,28.695926,"In segment 'At Risk | At Risk | Dual Product',..."
4,At Risk | At Risk | Dual Product,2,days_since_last_active,1.040279,"In segment 'At Risk | At Risk | Dual Product',..."
5,At Risk | At Risk | Dual Product,3,risk_score,0.387018,"In segment 'At Risk | At Risk | Dual Product',..."
6,At Risk | At Risk | Multi Product,1,total_amount,72.087252,In segment 'At Risk | At Risk | Multi Product'...
7,At Risk | At Risk | Multi Product,2,tenure_days,32.177943,In segment 'At Risk | At Risk | Multi Product'...
8,At Risk | At Risk | Multi Product,3,days_since_last_active,0.846733,In segment 'At Risk | At Risk | Multi Product'...
9,At Risk | At Risk | Single Product,1,days_since_last_active,2.491775,In segment 'At Risk | At Risk | Single Product...


## 11) Save Artifacts and Outputs

Exports created for app integration and partner reporting:
- calibrated model artifact
- feature list
- scored users
- segment risk table
- top-3 churn hypotheses

In [32]:
model_out = MODEL_DIR / f"fintech_{best_name}_calibrated_model.pkl"
features_out = MODEL_DIR / "fintech_features.pkl"

joblib.dump(calibrated_model, model_out)
joblib.dump(X.columns.tolist(), features_out)

scored.to_csv(PROJECT_ROOT / "data" / "fintech_scored_users.csv", index=False)
segment_scores.to_csv(PROJECT_ROOT / "data" / "fintech_segment_scores.csv", index=False)
hypotheses_df.to_csv(PROJECT_ROOT / "data" / "fintech_top3_churn_hypotheses.csv", index=False)

summary = {
    "best_base_model": best_name,
    "selected_threshold": selected_threshold,
    **metrics,
    "model_path": str(model_out),
    "features_path": str(features_out),
}

pd.DataFrame([summary])


,best_base_model,selected_threshold,roc_auc,accuracy,precision,recall,f1,tn,fp,fn,tp,model_path,features_path
0,xgboost,0.29,0.881703,0.8214,0.664776,0.770588,0.713782,5987,1123,663,2227,d:\Python\Data Science Projects\InsightForge A...,d:\Python\Data Science Projects\InsightForge A...
